# Qwen2.5-7B full fine-tune — RAG reliability judge (Method 1 / 2)
Full fine-tuning (not LoRA) of `Qwen/Qwen2.5-7B-Instruct`. Auto-detects GPU and
picks a memory strategy. Set `MODE` in the config cell to `"direct"` or `"marker"`.
Runs on Colab, Kaggle, Yandex DataSphere.

In [ ]:
# --- Install pinned deps (skip if already present) ---
import importlib, subprocess, sys

PKGS = [
    "transformers==4.46.3",
    "trl==0.12.2",
    "accelerate==1.1.1",
    "datasets==3.1.0",
    "peft==0.13.2",
    "bitsandbytes==0.44.1",
    "deepspeed==0.15.4",
    "sentencepiece",
]
def _pip(args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])
try:
    import trl, transformers  # noqa: F401
except Exception:
    _pip(PKGS)

# --- Detect platform ---
import os
if "google.colab" in sys.modules or os.path.exists("/content"):
    PLATFORM = "colab"
elif os.path.exists("/kaggle"):
    PLATFORM = "kaggle"
elif os.path.exists("/home/jupyter") or "DATASPHERE" in os.environ.get("HOSTNAME", "").upper():
    PLATFORM = "datasphere"
else:
    PLATFORM = "other"
print("platform:", PLATFORM)

In [ ]:
# --- Get the repo so training == inference format ---
REPO_URL = "https://github.com/<owner>/rag-reliability-judge.git"  # set to your fork/remote
REPO_DIR = "rag-reliability-judge"
import os, subprocess, sys

def _have_repo():
    try:
        import rag_reliability  # noqa: F401
        return True
    except Exception:
        return False

if not _have_repo():
    if not os.path.exists(REPO_DIR):
        try:
            subprocess.check_call(["git", "clone", "-q", REPO_URL, REPO_DIR])
        except Exception as e:
            print("clone failed, using inline fallback:", e)
    if os.path.isdir(REPO_DIR):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR])

try:
    from rag_reliability.nb_format import build_sft_messages
    from rag_reliability.dataset import load_jsonl, split_samples
    from rag_reliability.parsing import parse_prediction
    from rag_reliability.schema import RagSample
    from rag_reliability import metrics as M
    USING_REPO = True
except Exception as e:
    print("repo import failed -> inline fallback:", e)
    USING_REPO = False
    # Paste verbatim: nb_format.INLINE fallback (kept in sync by tests/test_nb_format.py)
    # NOTE for implementer: copy the bodies of build_direct_prompt/build_marker_prompt,
    # build_direct_target/build_marker_target, resolve_marker, RagSample, ALLOWED_MARKERS,
    # parse_prediction, load_jsonl, split_samples here. Source of truth: the repo modules.
    raise RuntimeError("Set REPO_URL to a reachable remote, or paste the inline fallback block.")

print("format source:", "repo" if USING_REPO else "inline")

In [ ]:
# --- Config: edit these ---
MODE = "direct"              # "direct" (Method 1) or "marker" (Method 2)
assert MODE in ("direct", "marker")
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

USE_REAL_DATA = False        # False -> dummy smoke set; True -> your organizers.jsonl
DATA_PATH = (f"{REPO_DIR}/data/dummy.jsonl" if not USE_REAL_DATA
             else "data/organizers.jsonl")

EPOCHS = 3
LR = 1e-5                    # full FT wants a smaller LR than LoRA
MAX_SEQ_LEN = 2048
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
SEED = 42

SAVE_TARGET = "auto"        # "auto" -> Drive on Colab / working dir elsewhere; or a path
QLORA_FALLBACK = False      # True only if hardware can't do full FT (NOT full FT)
OUTPUT_DIR = f"ft_{MODE}"
print(dict(MODE=MODE, model=BASE_MODEL, data=DATA_PATH, qlora=QLORA_FALLBACK))
